# xView classification: Swin
Original MSc coursework; Samuel confirms implementation in a group context.
2026 cleanup: identity alignment, saved reconstructed split, gradient remainder fix, safe image loading and prediction export.
Historical outputs are under evidence/; they do not describe this corrected implementation.


In [ ]:
import os
import sys
from pathlib import Path
from contextlib import nullcontext
sys.path.insert(0, str(Path.cwd().parent) if Path.cwd().name == "notebooks" else str(Path.cwd()))
from classification.core import records_from_annotations, historical_split, accumulation_group_size, accumulation_boundary, evaluate_records
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = Path(os.environ.get("XVIEW_DATA", ROOT / "data"))
RUN_DIR = ROOT / "runs" / "swin"
RUN_DIR.mkdir(parents=True, exist_ok=True)
def amp_context():
    return autocast() if device.type == "cuda" else nullcontext()

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import SwinForImageClassification, AutoImageProcessor
import torchvision.transforms as transforms
import cv2
import rasterio
import numpy as np
import warnings
import json
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
from tqdm.auto import tqdm
from torch.cuda.amp import GradScaler, autocast
import random
from PIL import Image

# Global Config
CATEGORIES = {0: 'Cargo plane', 1: 'Small car', 2: 'Bus', 3: 'Truck', 4: 'Motorboat',
              5: 'Fishing vessel', 6: 'Dump truck', 7: 'Excavator', 8: 'Building',
              9: 'Helipad', 10: 'Storage tank', 11: 'Shipping container', 12: 'Pylon'}
NUM_CLASSES = len(CATEGORIES)
IMG_SIZE = 224
BATCH_SIZE = 16  # Adjusted for Swin-Base (approx 8GB VRAM usage with FP16)
GRAD_ACCUM_STEPS = 2 # Effective Batch Size = 16 * 2 = 32
EPOCHS = 20
MODEL_NAME = 'microsoft/swin-base-patch4-window7-224'
LEARNING_RATE = 2e-5 # Lower LR for full fine-tuning

# 2026 explicit RNG seed; original training RNG state was not retained.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Setup Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
def load_geoimage(filename, size=(224, 224)):
    """Loads geospatial images and resizes them"""
    warnings.filterwarnings('ignore', category=rasterio.errors.NotGeoreferencedWarning)
    with rasterio.open(filename, 'r') as raster:
        img = np.moveaxis(raster.read(), 0, -1)
    if img.shape[-1] != 3:
        raise ValueError(f"Expected RGB image: {filename}")
    img = cv2.resize(img, size, interpolation=cv2.INTER_LINEAR)
    if img.dtype != np.uint8 and img.max() <= 1.0:
        img = img * 255.0
    return img.astype(np.float32)


In [ ]:
# ==========================================
# DATASET CLASS
# ==========================================

class SatelliteDataset(Dataset):
    def __init__(self, file_paths, labels, processor, img_size=224, augment=False):
        self.file_paths = file_paths
        self.labels = labels
        self.processor = processor
        self.img_size = img_size
        self.augment = augment

        if augment:
            self.augmentation = transforms.Compose([
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomVerticalFlip(p=0.5),
                transforms.RandomRotation(degrees=180),
                transforms.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
                transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
                transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            ])
        else:
            self.augmentation = None
        
    def __len__(self):
        return len(self.file_paths)
    
    def __getitem__(self, idx):
        img = load_geoimage(self.file_paths[idx], size=(self.img_size, self.img_size))
        img = img.astype(np.uint8)
        img_pil = Image.fromarray(img)
        
        if self.augmentation is not None:
            img_pil = self.augmentation(img_pil)
        
        # Swin Processor handles normalization
        inputs = self.processor(images=img_pil, return_tensors="pt")
        pixel_values = inputs['pixel_values'].squeeze(0)
        
        return pixel_values, self.labels[idx]

In [ ]:
# Reconstruct the historical split, joining labels by image identity.
json_file = DATA_ROOT / 'xview_ann_train.json'
import hashlib
manifest = json.loads((ROOT / 'evidence/split_manifest.json').read_text())
if hashlib.sha256(json_file.read_bytes()).hexdigest() != manifest['annotations_sha256']:
    raise ValueError('Annotation snapshot differs from the reconstructed historical split')
records = records_from_annotations(json.loads(json_file.read_text()))
train_records, valid_records = historical_split(records)
train_files = [str(DATA_ROOT / r['filename']) for r in train_records]
valid_files = [str(DATA_ROOT / r['filename']) for r in valid_records]
train_labels = [list(CATEGORIES.values()).index(r['label']) for r in train_records]
valid_labels = [list(CATEGORIES.values()).index(r['label']) for r in valid_records]
class_weights_array = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = torch.FloatTensor(class_weights_array).to(device)

print(f"✅ Data prepared: {len(train_files)} train, {len(valid_files)} valid")

# Create DataLoaders
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)

train_dataset = SatelliteDataset(train_files, train_labels, processor, img_size=IMG_SIZE, augment=True)
valid_dataset = SatelliteDataset(valid_files, valid_labels, processor, img_size=IMG_SIZE, augment=False)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
    num_workers=0, pin_memory=True
)
valid_loader = DataLoader(
    valid_dataset, batch_size=BATCH_SIZE, shuffle=False, 
    num_workers=0, pin_memory=True
)

In [ ]:
# ==========================================
# MODEL SETUP (FULL FINE-TUNING)
# ==========================================

print(f"\n🤖 Loading Swin Transformer: {MODEL_NAME}")

model = SwinForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    id2label={i: cat for i, cat in CATEGORIES.items()},
    label2id={cat: i for i, cat in CATEGORIES.items()},
    ignore_mismatched_sizes=True
)

# ✅ STRATEGY: Full Fine-Tuning (No Freezing)
# We want the model to fully adapt to satellite imagery features.
for param in model.parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"   • Trainable Parameters: {trainable_params:,}")
print(f"   • Strategy: Full Fine-Tuning")

model = model.to(device)

In [ ]:
# ==========================================
# OPTIMIZER & TRAINING UTILS
# ==========================================

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.05)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LEARNING_RATE * 10,
    epochs=EPOCHS,
    steps_per_epoch=(len(train_loader) + GRAD_ACCUM_STEPS - 1) // GRAD_ACCUM_STEPS,
    pct_start=0.1
)

scaler = GradScaler(enabled=device.type == "cuda") # For Mixed Precision

# Mixup/CutMix Functions
def mixup_data(x, y, alpha=0.2):
    if alpha > 0: lam = np.random.beta(alpha, alpha)
    else: lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def cutmix_data(x, y, alpha=1.0):
    if alpha > 0: lam = np.random.beta(alpha, alpha)
    else: lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)
    _, _, H, W = x.size()
    cut_rat = np.sqrt(1.0 - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)
    x[:, :, bby1:bby2, bbx1:bbx2] = x[index, :, bby1:bby2, bbx1:bbx2]
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (W * H))
    y_a, y_b = y, y[index]
    return x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [ ]:
# ==========================================
# TRAINING LOOP
# ==========================================

def train_epoch(model, loader, criterion, optimizer, scheduler, scaler, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    optimizer.zero_grad()
    
    pbar = tqdm(loader, desc='Training')
    for i, (pixel_values, labels) in enumerate(pbar):
        pixel_values, labels = pixel_values.to(device), labels.to(device)
        
        # Augmentation logic (CutMix/Mixup)
        rand_aug = np.random.random()
        use_mixup = False
        
        if rand_aug < 0.5: # 50% chance CutMix
            pixel_values, labels_a, labels_b, lam = cutmix_data(pixel_values, labels)
            use_mixup = True
        elif rand_aug < 0.8: # 30% chance Mixup
            pixel_values, labels_a, labels_b, lam = mixup_data(pixel_values, labels)
            use_mixup = True
            
        # Mixed Precision Forward Pass
        with amp_context():
            outputs = model(pixel_values).logits
            if use_mixup:
                loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
            else:
                loss = criterion(outputs, labels)
            group_size = accumulation_group_size(i, len(loader), GRAD_ACCUM_STEPS)
            loss = loss / group_size

        # Backward Pass with Scaler
        scaler.scale(loss).backward()
        
        if accumulation_boundary(i, len(loader), GRAD_ACCUM_STEPS):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            if scaler.get_scale() >= old_scale:
                scheduler.step()
            
        # Metrics (approximate for Mixup)
        running_loss += loss.item() * group_size * pixel_values.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        pbar.set_postfix({'loss': f'{running_loss/total:.4f}', 'acc': f'{100*correct/total:.2f}%'})
    
    return running_loss / total, 100 * correct / total

def validate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for pixel_values, labels in tqdm(loader, desc='Validation'):
            pixel_values, labels = pixel_values.to(device), labels.to(device)
            with amp_context():
                outputs = model(pixel_values).logits
                loss = criterion(outputs, labels)
            
            running_loss += loss.item() * pixel_values.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    return running_loss / total, 100 * correct / total

In [ ]:
config = dict(model=MODEL_NAME, epochs=EPOCHS, batch_size=BATCH_SIZE, accumulation=GRAD_ACCUM_STEPS, learning_rate=LEARNING_RATE, seed=SEED, annotations_sha256=manifest['annotations_sha256'], torch_version=torch.__version__)
(RUN_DIR / 'config.json').write_text(json.dumps(config, indent=2))

# ==========================================
# EXECUTE TRAINING
# ==========================================

best_val_acc = 0.0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print("\n🚀 Starting Swin Transformer Training...")

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    
    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, scheduler, scaler, device
    )
    
    val_loss, val_acc = validate_epoch(model, valid_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    
    print(f"   Train Loss: {train_loss:.4f} | Acc: {train_acc:.2f}%")
    print(f"   Val Loss:   {val_loss:.4f} | Acc: {val_acc:.2f}%")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), RUN_DIR / 'swin_best_model.pth')
        print(f"   ✅ New Best Model Saved! ({val_acc:.2f}%)")
        
with open(RUN_DIR / 'swin_history.json', 'w') as f:
    json.dump(history, f)

In [ ]:
# ==========================================
# TEST TIME AUGMENTATION (TTA)
# ==========================================

def predict_tta(model, dataset, device):
    model.eval()
    model.load_state_dict(torch.load(RUN_DIR / 'swin_best_model.pth', map_location=device, weights_only=True))
    
    predictions = []
    correct = 0
    total = 0
    
    # TTA Transforms: Original, Flip LR, Flip UD, Rotate 90
    transforms_tta = [
        lambda x: x,
        lambda x: transforms.functional.hflip(x),
        lambda x: transforms.functional.vflip(x),
        lambda x: transforms.functional.rotate(x, 90)
    ]
    
    print("\n🔍 Running TTA Evaluation...")
    
    for idx in tqdm(range(len(dataset))):
        # Load raw image again to apply TTA
        img_path = dataset.file_paths[idx]
        label = dataset.labels[idx]
        
        img = load_geoimage(img_path, size=(IMG_SIZE, IMG_SIZE))
        img = img.astype(np.uint8)
        img_pil = Image.fromarray(img)
        
        logits_sum = torch.zeros(NUM_CLASSES).to(device)
        
        for t in transforms_tta:
            aug_img = t(img_pil)
            inputs = processor(images=aug_img, return_tensors="pt")
            pixel_values = inputs['pixel_values'].to(device)
            
            with torch.no_grad():
                logits = model(pixel_values).logits
                logits_sum += logits.squeeze(0)
        
        predicted = torch.argmax(logits_sum).item()
        predictions.append(dict(image_id=valid_records[idx]['image_id'],
                                filename=valid_records[idx]['filename'],
                                predicted_label=CATEGORIES[predicted]))
        if predicted == label:
            correct += 1
        total += 1
        
    acc = 100 * correct / total
    print(f"\n🏆 Final TTA Accuracy: {acc:.2f}%")
    (RUN_DIR / 'predictions.json').write_text(json.dumps(predictions, indent=2))
    metrics = evaluate_records(valid_records, predictions)
    (RUN_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))
    provenance = dict(checkpoint_sha256=hashlib.sha256((RUN_DIR / 'swin_best_model.pth').read_bytes()).hexdigest(),
                      annotation_sha256=manifest['annotations_sha256'], split='validation',
                      model=MODEL_NAME, tta=['identity', 'hflip', 'vflip', 'rotate90'])
    (RUN_DIR / 'evaluation_manifest.json').write_text(json.dumps(provenance, indent=2))
    return metrics

predict_tta(model, valid_dataset, device)